# CEAP-360VR Dependence Contribution Analysis (Subject-dependent)

This supplementary notebook keeps the original MFMC architecture and training objective, but reduces the analysis to a single 80/20 held-out split by using the first split from the original protocol. It reports cyclic dependence contribution and pair-fusion gain over single-modality dependence directly inside the notebook.


## 1. Imports And Setup


In [ ]:
import os
import time
import subprocess
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from IPython.display import display

plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (8, 4.5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

FIGURE_SIZE = (8, 4.5)

def select_device(min_free_memory_mb=10000):
    """Prefer the CUDA GPU with the most free VRAM, otherwise fall back to CPU."""
    if not torch.cuda.is_available():
        print('CUDA is not available. Falling back to CPU; training will be slow.')
        return torch.device('cpu')

    try:
        query = [
            'nvidia-smi',
            '--query-gpu=index,memory.free,memory.total,memory.used',
            '--format=csv,noheader,nounits',
        ]
        output = subprocess.check_output(query, encoding='utf-8')
        gpu_stats = []
        for line in output.strip().splitlines():
            index, free_mem, total_mem, used_mem = [part.strip() for part in line.split(',')]
            gpu_stats.append({
                'index': int(index),
                'free': int(free_mem),
                'total': int(total_mem),
                'used': int(used_mem),
            })
        suitable_gpus = [gpu for gpu in gpu_stats if gpu['free'] >= min_free_memory_mb]
        selected = max(suitable_gpus or gpu_stats, key=lambda gpu: gpu['free'])
        device = torch.device(f"cuda:{selected['index']}")
        print(
            f"Using device: {device} "
            f"({selected['free']} MB free / {selected['total']} MB total, {selected['used']} MB used)"
        )
        return device
    except Exception as exc:
        print(f'nvidia-smi query failed ({exc}). Falling back to cuda:0.')
        return torch.device('cuda:0')


## 2. Configuration


In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

PROJECT_ROOT = os.environ.get('TAFFC_MFMC_ROOT', '/home/zhengdeyang/TAFFC_MFMC')
BASE_PATH = f'{PROJECT_ROOT}/MFMC/CEAP'
DATA_DIR = f'{BASE_PATH}/CEAP_Processed'

BATCH_SIZE = 200
GRADIENT_ACCUMULATION_STEPS = 1
TOTAL_ITERATIONS = 20001
EVAL_INTERVAL = 500

RANDOM_SEED = 42
LEARNING_RATE_ENCODER = 0.0003
LEARNING_RATE_CLASSIFIER = 0.0003
BETA1 = 0.9
BETA2 = 0.999
COV_BETA = 0.5

USE_CLASS_BALANCING = False
EVAL_BATCH_SIZE = 512
TRACE_EPS = 1e-6
DOMINANCE_THRESHOLD = 0.05

DATASET_NAME = 'CEAP-360VR'
PROTOCOL_NAME = 'Subject-dependent protocol'
SPLIT_TYPE = 'single 80/20 split'
SPLIT_ID = 1
PLOT_TITLE_CONTRIBUTION = 'CEAP-360VR Subject-Dependent Cyclic Dependence Contribution'
PLOT_TITLE_GAIN = 'CEAP-360VR Subject-Dependent Pair-Fusion Gain'
ENCODER_INPUT_KWARG = 'input_channels'

print('Configuration loaded successfully!')
print(f'Data directory: {DATA_DIR}')
print(f'Protocol: {PROTOCOL_NAME}')
print(f'Split type: {SPLIT_TYPE}')
print(f'Total iterations for this split: {TOTAL_ITERATIONS - 1}')
print(f'Batch size: {BATCH_SIZE}')


## 3. Original MFMC Loss And Projection Head


In [ ]:
# =============================================================================
# MFMC LOSS FUNCTIONS AND PROJECTION HEAD
# =============================================================================

def adaptive_estimation(v_t, beta, square_term, i):
    """
    Adaptive smoothing filter for estimating covariances

    Args:
        v_t: Previous covariance estimate
        beta: Exponential moving average coefficient
        square_term: Current covariance matrix
        i: Current iteration number

    Returns:
        Updated covariance estimate and bias-corrected estimate
    """
    v_t = beta * v_t + (1 - beta) * square_term.detach()
    return v_t, (v_t / (1 - beta ** i))

def MFMC_t_trace(x, y, track_cov, i, cov_beta=0.95):
    """
    Compute MFMC-T trace loss for a single pair of feature matrices

    This function implements the core MFMC trace loss that maximizes
    the correlation between features from different modalities.

    Args:
        x: Feature matrix from first modality [batch_size, feature_dim]
        y: Feature matrix from second modality [batch_size, feature_dim]
        track_cov: Dictionary containing tracking variables for covariances
        i: Current iteration number
        cov_beta: Exponential moving average coefficient for covariance tracking

    Returns:
        track_cov: Updated tracking dictionary
        loss: MFMC-T trace loss value
    """
    # Calculate auto-covariances and cross-covariance
    Rx = (x.T @ x) / x.shape[0]
    Ry = (y.T @ y) / y.shape[0]
    Pxy = (x.T @ y) / x.shape[0]

    # Add small epsilon for numerical stability
    eps = 1e-6
    Rx = Rx + torch.eye(Rx.shape[0]).to(Rx.device) * eps
    Ry = Ry + torch.eye(Ry.shape[0]).to(Ry.device) * eps

    # Update tracking estimates using adaptive estimation
    track_cov['Rx'], Rx_est = adaptive_estimation(track_cov['Rx'], cov_beta, Rx, i)
    track_cov['Ry'], Ry_est = adaptive_estimation(track_cov['Ry'], cov_beta, Ry, i)
    track_cov['Pxy'], Pxy_est = adaptive_estimation(track_cov['Pxy'], cov_beta, Pxy, i)

    # Compute matrix inverses
    Rx_est_inv = torch.inverse(Rx_est)
    Ry_est_inv = torch.inverse(Ry_est)

    # Compute MFMC-T trace cost (maximizes cross-modal correlation)
    cost = -Rx_est_inv @ Rx @ Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy @ Ry_est_inv @ Pxy_est.T \
           - Rx_est_inv @ Pxy_est @ Ry_est_inv @ Ry @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy.T

    # Return negative trace as the loss (to minimize)
    loss = -torch.trace(cost)

    return track_cov, loss

def tri_modal_projection_loss(fe1, fe2, fe3, proj12, proj23, proj13, trackers, step, cov_beta=0.5):
    """
    Compute the tri-modal projection loss for MFMC

    The loss function is: L = MFMC(E1, P23(E2⊕E3)) + MFMC(E2, P13(E1⊕E3)) + MFMC(E3, P12(E1⊕E2))

    This creates three pairwise correlations:
    - EDA features with projected BVP+SKT features
    - BVP features with projected EDA+SKT features  
    - SKT features with projected EDA+BVP features

    Args:
        fe1: EDA features [batch_size, 128]
        fe2: BVP features [batch_size, 128]
        fe3: SKT features [batch_size, 128]
        proj12: Projection head for E1⊕E2 → 128D
        proj23: Projection head for E2⊕E3 → 128D
        proj13: Projection head for E1⊕E3 → 128D
        trackers: Dictionary containing three covariance trackers
        step: Current iteration number
        cov_beta: Exponential moving average coefficient for covariance tracking

    Returns:
        trackers: Updated covariance trackers
        total_loss: Sum of three MFMC-T losses
    """
    # Concatenate features for projection (256D input for each projection head)
    concat_12 = torch.cat([fe1, fe2], dim=1)  # EDA + BVP → [batch_size, 256]
    concat_23 = torch.cat([fe2, fe3], dim=1)  # BVP + SKT → [batch_size, 256]
    concat_13 = torch.cat([fe1, fe3], dim=1)  # EDA + SKT → [batch_size, 256]

    # Apply projection heads to get 128D representations
    proj_12 = proj12(concat_12)  # [batch_size, 128]
    proj_23 = proj23(concat_23)  # [batch_size, 128]
    proj_13 = proj13(concat_13)  # [batch_size, 128]

    # Compute MFMC losses for each modality pairing
    # Loss 1: MFMC(EDA, P23(BVP⊕SKT))
    trackers['track_1_23'], loss1 = MFMC_t_trace(fe1, proj_23, trackers['track_1_23'], step, cov_beta)

    # Loss 2: MFMC(BVP, P13(EDA⊕SKT))
    trackers['track_2_13'], loss2 = MFMC_t_trace(fe2, proj_13, trackers['track_2_13'], step, cov_beta)

    # Loss 3: MFMC(SKT, P12(EDA⊕BVP))
    trackers['track_3_12'], loss3 = MFMC_t_trace(fe3, proj_12, trackers['track_3_12'], step, cov_beta)

    # Total tri-modal loss
    total_loss = loss1 + loss2 + loss3

    return trackers, total_loss

class ProjectionHead(nn.Module):
    """
    2-layer MLP projection head for MFMC

    Architecture: Linear(256→512) + BN + ReLU → Linear(512→128) + BN

    Takes concatenated features from two modalities (256D) and projects
    to a common 128D representation space for correlation computation.
    """
    def __init__(self, input_dim=256, hidden_dim=512, output_dim=128):
        super(ProjectionHead, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        return x

print("MFMC loss functions and projection head defined successfully!")


## 4. Original Model Definitions


In [ ]:
# =============================================================================
# NEURAL NETWORK ARCHITECTURES
# =============================================================================


# Define network for temporal network: 
class NETWORK_F_MLP(nn.Module):
    """
    Multi-layer perceptron for feature transformation
    """
    def __init__(self, input_dim=784, hidden_dim=200, out_dim=200, num_layers=2):
        super(NETWORK_F_MLP, self).__init__()
        self.dim = out_dim
        self.num_layers = num_layers

        self.fc_list = []
        self.bn_list = []

        # First layer
        self.fc_list.append(nn.Linear(input_dim, hidden_dim, bias=True))
        self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        # Hidden layers
        for i in range(self.num_layers - 1):
            self.fc_list.append(nn.Linear(hidden_dim, hidden_dim, bias=True))
            self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        self.fc_list = nn.ModuleList(self.fc_list)
        self.bn_list = nn.ModuleList(self.bn_list)

        # Final layer
        self.fc_final = nn.Linear(hidden_dim, out_dim, bias=True)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)

        for i in range(self.num_layers):
            x = self.fc_list[i](x)
            x = torch.relu(x)
            x = self.bn_list[i](x)

        x = self.fc_final(x)
        x = torch.sigmoid(x)
        return x

# Define network for channel network: 
class Advanced1DCNN_channel(nn.Module):
    """
    Advanced 1D CNN for processing physiological signals

    This network processes multi-channel time series data (EDA, BVP, SKT)
    and extracts meaningful features for emotion recognition.
    """
    def __init__(self, input_channels=1, num_classes=128, input_size=1280):
        super(Advanced1DCNN_channel, self).__init__()

        # Convolutional layers with increasing depth
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=11, padding=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        self.conv4 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=11, padding=5),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        # Calculate feature size after convolutions
        feat_size = input_size // (4 * 4 * 4 * 4)  # 4 max-pooling layers

        # Fully connected layers
        self.fc1 = nn.Sequential(
            nn.Linear(256 * feat_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
        )

        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
        )

        self.fc3 = nn.Linear(512, num_classes)

        # Multi-layer perceptron for final feature processing
        self.MLP = NETWORK_F_MLP(
            input_dim=128 * input_channels, 
            hidden_dim=4000, 
            out_dim=num_classes, 
            num_layers=1
        )

    def forward(self, x):
        batch_size, channels = x.shape[0], x.shape[1]

        # Process each channel separately through CNN
        x = x.unsqueeze(2)  # Add temporal dimension
        x = x.flatten(0, 1)  # Combine batch and channel dimensions

        # Convolutional feature extraction
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)

        # Flatten and process through FC layers
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)

        # Reshape and process through MLP
        out = out.reshape(batch_size, channels, -1)
        out = out.flatten(-2, -1)
        out = self.MLP(out)

        return out

# Define network for classification:
class ComplexClassifier(nn.Module):
    """
    Multi-layer classifier for emotion recognition

    Takes 128D features from any modality and predicts emotion class.
    """
    def __init__(self, dim_features=128, num_classes=4):
        super(ComplexClassifier, self).__init__()

        self.fc1 = nn.Linear(dim_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)  # No activation - CrossEntropyLoss includes softmax
        return x

print("Neural network architectures defined successfully!")


## 5. Data Loading And Single-Split Construction


In [ ]:
# =============================================================================
# DATA LOADING AND SINGLE SUBJECT-DEPENDENT SPLIT CONSTRUCTION
# =============================================================================

print('Loading CEAP-360VR dataset...')
print('This supplement uses the first split from the original subject-dependent 5-fold protocol.')
print('Modalities: EDA, BVP, SKT')

try:
    print(f'Loading data from: {DATA_DIR}')
    subject = np.load(f'{DATA_DIR}/subject.npy')
    emotion_labels = np.load(f'{DATA_DIR}/emotion_labels.npy')
    eda_data = np.load(f'{DATA_DIR}/eda_data.npy')
    bvp_data = np.load(f'{DATA_DIR}/bvp_data.npy')
    skt_data = np.load(f'{DATA_DIR}/skt_data.npy')

    subject = torch.from_numpy(subject).long()
    emotion_labels = torch.from_numpy(emotion_labels).long()
    eda_data = torch.from_numpy(eda_data).float()
    bvp_data = torch.from_numpy(bvp_data).float()
    skt_data = torch.from_numpy(skt_data).float()

    print('\nData loaded successfully!')
    print(f'- Total samples: {eda_data.shape[0]}')
    print(f'- EDA shape: {tuple(eda_data.shape)}')
    print(f'- BVP shape: {tuple(bvp_data.shape)}')
    print(f'- SKT shape: {tuple(skt_data.shape)}')
    print(f'- Number of unique subjects: {len(torch.unique(subject))}')
    print(f'- Number of emotion classes: {len(torch.unique(emotion_labels))}')

    class_names = ['Low V-Low A (Sad)', 'Low V-High A (Angry)', 'High V-Low A (Calm)', 'High V-High A (Happy)']
    class_counts = torch.bincount(emotion_labels)
    print('\nEmotion class distribution:')
    for idx, (name, count) in enumerate(zip(class_names, class_counts)):
        print(f'  Class {idx} - {name}: {count} samples ({count / len(emotion_labels) * 100:.1f}%)')

    from sklearn.model_selection import StratifiedKFold

    indices = np.arange(eda_data.shape[0])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    first_train_indices, first_test_indices = next(iter(skf.split(indices, emotion_labels.numpy())))
    split_info = {
        'split_id': SPLIT_ID,
        'train_indices': first_train_indices,
        'test_indices': first_test_indices,
        'train_size': len(first_train_indices),
        'test_size': len(first_test_indices),
    }

    print('\nSingle split ready: using fold 1 from the original 5-fold generator.')
    print(f"Train samples={split_info['train_size']} | Test samples={split_info['test_size']}")
    data_loaded = True
except FileNotFoundError as exc:
    print(f'Error loading data: {exc}')
    data_loaded = False
except Exception as exc:
    print(f'Unexpected error: {exc}')
    data_loaded = False


## 6. Stable Dependence Utilities


In [ ]:
# =============================================================================
# STABLE DEPENDENCE CONTRIBUTION AND PAIR-FUSION GAIN UTILITIES
# =============================================================================

INTERACTION_LABELS = {
    '12_to_3': '(EDA,BVP)->SKT',
    '13_to_2': '(EDA,SKT)->BVP',
    '23_to_1': '(BVP,SKT)->EDA',
}

SINGLE_BASELINE_LABELS = {
    '12_to_3': 'EDA->SKT vs BVP->SKT',
    '13_to_2': 'EDA->BVP vs SKT->BVP',
    '23_to_1': 'BVP->EDA vs SKT->EDA',
}

def _symmetrize(matrix):
    return 0.5 * (matrix + matrix.T)

def positive_trace_dependence_stable(x, y, eps=1e-6, clamp_small_negative=True):
    x = x.to(dtype=torch.float64)
    y = y.to(dtype=torch.float64)

    n = x.shape[0]
    if n < 2:
        raise RuntimeError('Need at least two samples to estimate dependence.')

    rx = (x.T @ x) / n
    ry = (y.T @ y) / n
    pxy = (x.T @ y) / n

    rx = _symmetrize(rx)
    ry = _symmetrize(ry)

    eye_x = torch.eye(rx.shape[0], device=rx.device, dtype=rx.dtype)
    eye_y = torch.eye(ry.shape[0], device=ry.device, dtype=ry.dtype)

    last_error = None
    for factor in [1.0, 10.0, 100.0, 1000.0, 10000.0]:
        ridge = eps * factor
        try:
            rx_reg = rx + ridge * eye_x
            ry_reg = ry + ridge * eye_y

            lx = torch.linalg.cholesky(rx_reg)
            ly = torch.linalg.cholesky(ry_reg)

            a = torch.cholesky_solve(pxy, lx)
            c = torch.cholesky_solve(pxy.T, ly).T
            trace_value = torch.sum(a * c)

            if not torch.isfinite(trace_value):
                raise RuntimeError(f'Non-finite trace value with ridge={ridge}')

            value = trace_value.item()
            if value < 0:
                if clamp_small_negative and value > -1e-8:
                    return torch.zeros((), device=x.device, dtype=torch.float64)
                raise RuntimeError(
                    f'Negative trace value {value} with ridge={ridge}. '
                    'This indicates numerical instability.'
                )

            return trace_value
        except RuntimeError as exc:
            last_error = exc
            continue

    raise RuntimeError(f'Failed stable trace computation. Last error: {last_error}')

def build_encoder(num_input_channels, input_size, device):
    encoder_kwargs = {
        ENCODER_INPUT_KWARG: num_input_channels,
        'num_classes': 128,
        'input_size': input_size,
    }
    return Advanced1DCNN_channel(**encoder_kwargs).to(device)

def create_split_models(device):
    NET_EDA = build_encoder(eda_data.shape[1], eda_data.shape[2], device)
    NET_BVP = build_encoder(bvp_data.shape[1], bvp_data.shape[2], device)
    NET_SKT = build_encoder(skt_data.shape[1], skt_data.shape[2], device)

    proj_12 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)
    proj_23 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)
    proj_13 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)

    num_classes = len(torch.unique(emotion_labels))
    classifier = ComplexClassifier(dim_features=128, num_classes=num_classes).to(device)
    return NET_EDA, NET_BVP, NET_SKT, proj_12, proj_23, proj_13, classifier

def initialize_trackers(feature_dim, device):
    return {
        'track_1_23': {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
        },
        'track_2_13': {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
        },
        'track_3_12': {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
        },
    }

def evaluate_classifier_accuracy(NET_EDA, classifier, test_eda, test_labels, device, batch_size=100):
    NET_EDA.eval()
    classifier.eval()

    correct = 0
    total = 0
    num_batches = (len(test_eda) + batch_size - 1) // batch_size

    with torch.no_grad():
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, len(test_eda))
            test_batch = test_eda[start_idx:end_idx].to(device)
            labels_batch = test_labels[start_idx:end_idx].to(device)
            outputs = classifier(NET_EDA(test_batch))
            _, predicted = torch.max(outputs.data, 1)
            total += labels_batch.size(0)
            correct += (predicted == labels_batch).sum().item()

    return correct / total

def evaluate_dependence_metrics(models, heldout_data, device, eval_batch_size=512):
    NET_EDA = models['NET_EDA']
    NET_BVP = models['NET_BVP']
    NET_SKT = models['NET_SKT']
    proj_12 = models['proj_12']
    proj_13 = models['proj_13']
    proj_23 = models['proj_23']

    modules = [NET_EDA, NET_BVP, NET_SKT, proj_12, proj_13, proj_23]
    previous_modes = [module.training for module in modules]
    for module in modules:
        module.eval()

    test_eda = heldout_data['eda']
    test_bvp = heldout_data['bvp']
    test_skt = heldout_data['skt']
    num_samples = len(test_eda)
    num_batches = (num_samples + eval_batch_size - 1) // eval_batch_size

    fe1_chunks, fe2_chunks, fe3_chunks = [], [], []
    e12_chunks, e13_chunks, e23_chunks = [], [], []

    with torch.no_grad():
        for batch_idx in range(num_batches):
            start_idx = batch_idx * eval_batch_size
            end_idx = min((batch_idx + 1) * eval_batch_size, num_samples)

            batch_1 = test_eda[start_idx:end_idx].to(device)
            batch_2 = test_bvp[start_idx:end_idx].to(device)
            batch_3 = test_skt[start_idx:end_idx].to(device)

            fe1 = NET_EDA(batch_1)
            fe2 = NET_BVP(batch_2)
            fe3 = NET_SKT(batch_3)

            e12 = proj_12(torch.cat([fe1, fe2], dim=1))
            e13 = proj_13(torch.cat([fe1, fe3], dim=1))
            e23 = proj_23(torch.cat([fe2, fe3], dim=1))

            fe1_chunks.append(fe1.detach().cpu())
            fe2_chunks.append(fe2.detach().cpu())
            fe3_chunks.append(fe3.detach().cpu())
            e12_chunks.append(e12.detach().cpu())
            e13_chunks.append(e13.detach().cpu())
            e23_chunks.append(e23.detach().cpu())

    for module, was_training in zip(modules, previous_modes):
        module.train(was_training)

    fe1_all = torch.cat(fe1_chunks, dim=0)
    fe2_all = torch.cat(fe2_chunks, dim=0)
    fe3_all = torch.cat(fe3_chunks, dim=0)
    e12_all = torch.cat(e12_chunks, dim=0)
    e13_all = torch.cat(e13_chunks, dim=0)
    e23_all = torch.cat(e23_chunks, dim=0)

    fused_12_to_3 = positive_trace_dependence_stable(e12_all, fe3_all, eps=TRACE_EPS).item()
    fused_13_to_2 = positive_trace_dependence_stable(e13_all, fe2_all, eps=TRACE_EPS).item()
    fused_23_to_1 = positive_trace_dependence_stable(e23_all, fe1_all, eps=TRACE_EPS).item()

    total_raw = fused_12_to_3 + fused_13_to_2 + fused_23_to_1
    if total_raw <= 0:
        raise RuntimeError(f'Expected positive total dependence, got {total_raw}.')

    single_1_to_3 = positive_trace_dependence_stable(fe1_all, fe3_all, eps=TRACE_EPS).item()
    single_2_to_3 = positive_trace_dependence_stable(fe2_all, fe3_all, eps=TRACE_EPS).item()
    single_1_to_2 = positive_trace_dependence_stable(fe1_all, fe2_all, eps=TRACE_EPS).item()
    single_3_to_2 = positive_trace_dependence_stable(fe3_all, fe2_all, eps=TRACE_EPS).item()
    single_2_to_1 = positive_trace_dependence_stable(fe2_all, fe1_all, eps=TRACE_EPS).item()
    single_3_to_1 = positive_trace_dependence_stable(fe3_all, fe1_all, eps=TRACE_EPS).item()

    best_single_12_to_3 = max(single_1_to_3, single_2_to_3)
    best_single_13_to_2 = max(single_1_to_2, single_3_to_2)
    best_single_23_to_1 = max(single_2_to_1, single_3_to_1)

    absolute_gain_12_to_3 = fused_12_to_3 - best_single_12_to_3
    absolute_gain_13_to_2 = fused_13_to_2 - best_single_13_to_2
    absolute_gain_23_to_1 = fused_23_to_1 - best_single_23_to_1

    relative_gain_12_to_3 = absolute_gain_12_to_3 / (best_single_12_to_3 + 1e-12)
    relative_gain_13_to_2 = absolute_gain_13_to_2 / (best_single_13_to_2 + 1e-12)
    relative_gain_23_to_1 = absolute_gain_23_to_1 / (best_single_23_to_1 + 1e-12)

    contribution_rows = [
        {'interaction': INTERACTION_LABELS['12_to_3'], 'raw_trace': fused_12_to_3, 'normalized_contribution': fused_12_to_3 / total_raw},
        {'interaction': INTERACTION_LABELS['13_to_2'], 'raw_trace': fused_13_to_2, 'normalized_contribution': fused_13_to_2 / total_raw},
        {'interaction': INTERACTION_LABELS['23_to_1'], 'raw_trace': fused_23_to_1, 'normalized_contribution': fused_23_to_1 / total_raw},
    ]

    gain_rows = [
        {
            'interaction': INTERACTION_LABELS['12_to_3'],
            'fused_pair_trace': fused_12_to_3,
            'best_single_trace': best_single_12_to_3,
            'absolute_gain': absolute_gain_12_to_3,
            'relative_gain': relative_gain_12_to_3,
            'relative_gain_percent': 100.0 * relative_gain_12_to_3,
        },
        {
            'interaction': INTERACTION_LABELS['13_to_2'],
            'fused_pair_trace': fused_13_to_2,
            'best_single_trace': best_single_13_to_2,
            'absolute_gain': absolute_gain_13_to_2,
            'relative_gain': relative_gain_13_to_2,
            'relative_gain_percent': 100.0 * relative_gain_13_to_2,
        },
        {
            'interaction': INTERACTION_LABELS['23_to_1'],
            'fused_pair_trace': fused_23_to_1,
            'best_single_trace': best_single_23_to_1,
            'absolute_gain': absolute_gain_23_to_1,
            'relative_gain': relative_gain_23_to_1,
            'relative_gain_percent': 100.0 * relative_gain_23_to_1,
        },
    ]

    return {
        'contribution_rows': contribution_rows,
        'gain_rows': gain_rows,
        'evaluated_samples': num_samples,
        'evaluated_batches': num_batches,
    }

def summarize_dominance(contribution_df):
    ordered = contribution_df.sort_values('normalized_contribution', ascending=False).reset_index(drop=True)
    dominant_interaction = ordered.loc[0, 'interaction']
    dominant_value = float(ordered.loc[0, 'normalized_contribution'])
    second_largest_value = float(ordered.loc[1, 'normalized_contribution'])
    dominance_margin = dominant_value - second_largest_value
    if dominance_margin < DOMINANCE_THRESHOLD:
        dominant_interaction = 'balanced'
    return dominant_interaction, dominant_value, second_largest_value, dominance_margin


## 7. Single-Split Training And Evaluation


In [ ]:
# =============================================================================
# SINGLE-SPLIT TRAINING AND DEPENDENCE EVALUATION
# =============================================================================

if not data_loaded:
    raise RuntimeError('Data loading failed; cannot start training.')

device = select_device()
train_indices = split_info['train_indices']
test_indices = split_info['test_indices']

train_eda = eda_data[train_indices]
test_eda = eda_data[test_indices]
train_bvp = bvp_data[train_indices]
test_bvp = bvp_data[test_indices]
train_skt = skt_data[train_indices]
test_skt = skt_data[test_indices]
train_labels = emotion_labels[train_indices]
test_labels = emotion_labels[test_indices]

num_classes = len(torch.unique(emotion_labels))
train_class_counts = torch.bincount(train_labels, minlength=num_classes)
if USE_CLASS_BALANCING:
    class_weights = 1.0 / train_class_counts.float()
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
else:
    criterion = nn.CrossEntropyLoss()

NET_EDA, NET_BVP, NET_SKT, proj_12, proj_23, proj_13, classifier = create_split_models(device)

all_feature_params = (
    list(NET_EDA.parameters())
    + list(NET_BVP.parameters())
    + list(NET_SKT.parameters())
    + list(proj_12.parameters())
    + list(proj_23.parameters())
    + list(proj_13.parameters())
)

optimizer_features = optim.Adam(
    all_feature_params,
    lr=LEARNING_RATE_ENCODER,
    betas=(BETA1, BETA2),
    amsgrad=True,
)
optimizer_classifier = optim.Adam(
    classifier.parameters(),
    lr=LEARNING_RATE_CLASSIFIER,
    betas=(BETA1, BETA2),
    amsgrad=True,
)

trackers = initialize_trackers(feature_dim=128, device=device)
training_costs = []
classifier_losses = []
test_accuracies = []
best_accuracy = 0.0
split_start_time = time.time()

print(f"\n{'=' * 70}")
print(f'Training single split {SPLIT_ID}')
print(f'Train samples: {len(train_indices)} | Test samples: {len(test_indices)}')
if 'train_subjects' in split_info and 'test_subjects' in split_info:
    print(f"Train subjects: {split_info['train_subjects']}")
    print(f"Test subjects: {split_info['test_subjects']}")
print(f"{'=' * 70}")

for iteration in range(1, TOTAL_ITERATIONS):
    total_mfmc_loss = 0.0
    total_classifier_loss = 0.0

    optimizer_features.zero_grad()
    for _ in range(GRADIENT_ACCUMULATION_STEPS):
        batch_indices = torch.randint(0, len(train_eda), (BATCH_SIZE,))
        input_1 = train_eda[batch_indices].to(device)
        input_2 = train_bvp[batch_indices].to(device)
        input_3 = train_skt[batch_indices].to(device)

        feature_1 = NET_EDA(input_1)
        feature_2 = NET_BVP(input_2)
        feature_3 = NET_SKT(input_3)

        trackers, mfmc_loss = tri_modal_projection_loss(
            feature_1,
            feature_2,
            feature_3,
            proj_12,
            proj_23,
            proj_13,
            trackers,
            iteration,
            COV_BETA,
        )
        (mfmc_loss / GRADIENT_ACCUMULATION_STEPS).backward()
        total_mfmc_loss += mfmc_loss.item() / GRADIENT_ACCUMULATION_STEPS
    optimizer_features.step()

    optimizer_classifier.zero_grad()
    for _ in range(GRADIENT_ACCUMULATION_STEPS):
        batch_indices = torch.randint(0, len(train_eda), (BATCH_SIZE,))
        input_1 = train_eda[batch_indices].to(device)
        with torch.no_grad():
            feature_1 = NET_EDA(input_1)
        labels_batch = train_labels[batch_indices].to(device)
        output_class = classifier(feature_1.detach())
        classifier_loss = criterion(output_class, labels_batch)
        (classifier_loss / GRADIENT_ACCUMULATION_STEPS).backward()
        total_classifier_loss += classifier_loss.item() / GRADIENT_ACCUMULATION_STEPS
    optimizer_classifier.step()

    training_costs.append(total_mfmc_loss)
    classifier_losses.append(total_classifier_loss)

    if iteration % 500 == 0:
        print(
            f'Split {SPLIT_ID} - Iter {iteration:5d} | '
            f'MFMC: {total_mfmc_loss:.6f} | '
            f'Classifier: {total_classifier_loss:.6f}'
        )

    if iteration % EVAL_INTERVAL == 0:
        split_accuracy = evaluate_classifier_accuracy(
            NET_EDA,
            classifier,
            test_eda,
            test_labels,
            device,
        )
        test_accuracies.append(split_accuracy)
        best_accuracy = max(best_accuracy, split_accuracy)
        NET_EDA.train()
        classifier.train()

analysis_results = evaluate_dependence_metrics(
    models={
        'NET_EDA': NET_EDA,
        'NET_BVP': NET_BVP,
        'NET_SKT': NET_SKT,
        'proj_12': proj_12,
        'proj_13': proj_13,
        'proj_23': proj_23,
    },
    heldout_data={
        'eda': test_eda,
        'bvp': test_bvp,
        'skt': test_skt,
    },
    device=device,
    eval_batch_size=EVAL_BATCH_SIZE,
)

split_duration = time.time() - split_start_time
contribution_df = pd.DataFrame(analysis_results['contribution_rows'])
gain_df = pd.DataFrame(analysis_results['gain_rows'])
dominant_interaction, dominant_value, second_largest_value, dominance_margin = summarize_dominance(contribution_df)

config_display = {
    'dataset': DATASET_NAME,
    'protocol': PROTOCOL_NAME,
    'split_type': SPLIT_TYPE,
    'split_id': SPLIT_ID,
    'training_samples': len(train_indices),
    'test_samples': len(test_indices),
    'evaluated_samples': analysis_results['evaluated_samples'],
    'evaluated_batches': analysis_results['evaluated_batches'],
    'best_eda_classifier_accuracy': best_accuracy,
    'training_time_seconds': split_duration,
}
if 'train_subjects' in split_info:
    config_display['train_subjects'] = split_info['train_subjects']
    config_display['test_subjects'] = split_info['test_subjects']


## 8. Inline Results, Tables, Plots, And Interpretation


In [ ]:
# =============================================================================
# INLINE RESULTS TABLES, PLOTS, AND INTERPRETATION
# =============================================================================

print('Configuration for this supplementary run:')
for key, value in config_display.items():
    print(f'- {{key}}: {{value}}')

print('\nCyclic dependence contribution table (single 80/20 split):')
display(contribution_df)

print('\nPair-fusion gain table (single 80/20 split):')
display(gain_df)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.bar(contribution_df['interaction'], contribution_df['normalized_contribution'], color=['#4C78A8', '#F58518', '#54A24B'], edgecolor='black', linewidth=0.8)
ax.set_ylabel('Normalized contribution')
ax.set_title(PLOT_TITLE_CONTRIBUTION)
ax.set_ylim(0, max(0.6, float(contribution_df['normalized_contribution'].max()) + 0.05))
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.bar(gain_df['interaction'], gain_df['relative_gain_percent'], color=['#4C78A8', '#F58518', '#54A24B'], edgecolor='black', linewidth=0.8)
ax.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
ax.set_ylabel('Relative pair-fusion gain (%)')
ax.set_title(PLOT_TITLE_GAIN)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

positive_gain_rows = gain_df[gain_df['relative_gain'] > 0]
non_positive_gain_rows = gain_df[gain_df['relative_gain'] <= 0]

if dominance_margin >= DOMINANCE_THRESHOLD:
    cyclic_text = (
        f"Cyclic contribution analysis for this single split identifies {{dominant_interaction}} as the dominant "
        f"pair-to-third term, with a dominance margin of {{dominance_margin:.3f}} over the second-largest contribution."
    )
else:
    cyclic_text = (
        'Cyclic contribution analysis for this single split is relatively balanced, suggesting that MFMC '
        'does not rely on one strongly dominant pair-to-third term.'
    )

if len(positive_gain_rows) == len(gain_df):
    gain_text = (
        'All three directions show positive pair-fusion gain, indicating that the fused pair representations '
        'have stronger dependence with the third modality than the strongest corresponding single-modality baseline.'
    )
elif len(positive_gain_rows) >= 2:
    gain_text = (
        'Most directions show positive pair-fusion gain. Positive directions: '
        + ', '.join(positive_gain_rows['interaction'].tolist())
        + '. This suggests that joint pair representations often strengthen dependence with the third modality beyond the best single-modality baseline.'
    )
elif len(positive_gain_rows) == 1:
    gain_text = (
        'Pair-fusion gain is mixed. Only '
        + positive_gain_rows['interaction'].iloc[0]
        + ' shows positive pair-fusion gain, so the learned higher-order structure appears direction-specific rather than uniformly strengthened.'
    )
else:
    gain_text = (
        'Pair-fusion gain is near zero or negative in all directions, suggesting that the learned higher-order '
        'structure is more distributed and partially redundant than dominated by a strongly gainful pair-to-third interaction.'
    )

print('\nCombined interpretation:')
print(cyclic_text)
print(gain_text)
print(
    'Cyclic contribution analysis indicates whether MFMC relies on a single dominant pair-to-third term or '
    'distributes dependence across cyclic terms. Pair-fusion gain further compares each fused pair representation '
    'with the strongest corresponding single-modality baseline. Positive pair-fusion gain suggests that the fused '
    'pair representation captures additional joint dependence with the third modality beyond either single modality alone.'
)
